# Analisis Smart Office: okupansi, lintasan, deteksi, dan evaluasi

Notebook ini menganalisis ekspor dari dashboard Smart Office. Di dashboard, pilih **Unduh data untuk Colab**, lalu unggah ZIP tersebut pada sel berikut.

Data tracking memakai ID tracker centroid ringan dan posisi yang dinormalisasi (0 sampai 1). Sampel evaluasi mencakup objek dan confidence untuk analisis overlay. Confidence adalah keyakinan prediksi, bukan akurasi. Metrik akurasi hitung hanya tersedia setelah jumlah aktual dikonfirmasi. Notebook ini tidak melatih ulang bobot YOLO; pelatihan khusus memerlukan gambar dan anotasi kotak kelas objek.

In [ ]:
from google.colab import files
import os, zipfile, glob, json
uploaded = files.upload()  # Pilih smartoffice_colab_export.zip dari dashboard
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Unggah file ZIP hasil ekspor dashboard.')
extract_dir = '/content/smartoffice_export'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(extract_dir)
print('File diekstrak ke:', extract_dir)
print('\\n'.join(os.path.relpath(p, extract_dir) for p in glob.glob(extract_dir + '/**/*', recursive=True) if os.path.isfile(p))[:3000])

In [ ]:
import pandas as pd

def read_export_csv(name):
    path = os.path.join(extract_dir, name)
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

occupancy = read_export_csv('occupancy.csv')
tracks = read_export_csv('tracks.csv')
evaluation = read_export_csv('evaluation_samples.csv')
for frame in (occupancy, tracks, evaluation):
    if 'sampled_at' in frame: frame['sampled_at'] = pd.to_datetime(frame['sampled_at'], errors='coerce')
    if 'captured_at' in frame: frame['captured_at'] = pd.to_datetime(frame['captured_at'], errors='coerce')
print(f'Baris okupansi: {len(occupancy):,} | titik lintasan: {len(tracks):,} | sampel evaluasi: {len(evaluation):,}')
display(evaluation.head())
audio = read_export_csv('audio_samples.csv')
quality_events = read_export_csv('quality_events.csv')
zone_visits = read_export_csv('zone_visits.csv')
if 'sampled_at' in audio: audio['sampled_at'] = pd.to_datetime(audio['sampled_at'], errors='coerce')
if 'observed_at' in quality_events: quality_events['observed_at'] = pd.to_datetime(quality_events['observed_at'], errors='coerce')
if 'entered_at' in zone_visits: zone_visits['entered_at'] = pd.to_datetime(zone_visits['entered_at'], errors='coerce')
if 'exited_at' in zone_visits: zone_visits['exited_at'] = pd.to_datetime(zone_visits['exited_at'], errors='coerce')
print(f'Segmen audio: {len(audio):,} | event frame gelap: {len(quality_events):,}')


## 1. Tren jumlah orang
Setiap titik berasal dari pembacaan okupansi aplikasi. ID sesi memisahkan satu kali sesi kamera dari sesi lainnya.

In [ ]:
import matplotlib.pyplot as plt
if occupancy.empty:
    print('Belum ada data okupansi. Jalankan kamera dan kumpulkan data terlebih dahulu.')
else:
    occupancy = occupancy.dropna(subset=['sampled_at']).sort_values('sampled_at')
    plt.figure(figsize=(14, 4))
    for session, group in occupancy.groupby('session_id'):
        plt.plot(group['sampled_at'], group['person_count'], alpha=.8, label=str(session))
    plt.title('Jumlah orang dari waktu ke waktu'); plt.xlabel('Waktu'); plt.ylabel('Jumlah orang')
    plt.grid(alpha=.25)
    if occupancy['session_id'].nunique() <= 8: plt.legend(title='Sesi', loc='upper left', bbox_to_anchor=(1,1))
    plt.tight_layout(); plt.show()

## Confidence dan objek terdeteksi
Sampel evaluasi membawa daftar kelas objek dan confidence dari overlay kamera. Confidence bukan akurasi; bagian ini merangkum prediksi tersimpan, tanpa menganggapnya label benar.

In [ ]:
import re
detections = []
if not evaluation.empty and {'objects_json','captured_at'}.issubset(evaluation.columns):
    for _, row in evaluation.iterrows():
        try:
            objects = json.loads(row['objects_json']) if pd.notna(row['objects_json']) else []
        except (TypeError, json.JSONDecodeError):
            objects = []
        for item in objects if isinstance(objects, list) else []:
            match = re.match(r'^(.+?)\s*\((\d+(?:\.\d+)?)%\)$', str(item))
            if match:
                detections.append({'captured_at': row['captured_at'], 'class': match.group(1), 'confidence_percent': float(match.group(2))})
detection_df = pd.DataFrame(detections)
if detection_df.empty:
    print('Belum ada deteksi objek pada sampel evaluasi. Pastikan objek didukung model dan confidence cukup rendah.')
else:
    summary = detection_df.groupby('class').agg(detections=('class','size'), mean_confidence=('confidence_percent','mean')).sort_values('detections', ascending=False)
    display(summary.round(1))
    ax = summary['detections'].plot(kind='bar', figsize=(9,4), color='#f28c45', title='Jumlah deteksi per kelas')
    ax.set_ylabel('Jumlah prediksi tersimpan'); ax.set_xlabel('Kelas'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
    print('Rata-rata confidence semua objek: {:.1f}%'.format(detection_df['confidence_percent'].mean()))

Pengaturan confidence dan ROI yang diubah di dashboard berlaku di runtime aplikasi. Ekspor saat ini tidak menyertakan riwayat perubahan pengaturan; analisis di atas memakai deteksi yang tersimpan pada sampel gambar evaluasi.

## 2. Heatmap posisi orang
Koordinat lintasan sudah dinormalisasi terhadap ukuran frame. Sumbu X menunjukkan lebar kamera dan sumbu Y menunjukkan tinggi kamera.

In [ ]:
import numpy as np
if tracks.empty:
    print('Belum ada titik tracking. Pastikan kamera aktif dan versi aplikasi dengan tracker centroid sedang berjalan.')
else:
    x = tracks['x_norm'].clip(0, 1).to_numpy()
    y = tracks['y_norm'].clip(0, 1).to_numpy()
    heat, xedges, yedges = np.histogram2d(x, y, bins=(32, 24), range=[[0,1],[0,1]])
    plt.figure(figsize=(8, 5))
    plt.imshow(heat.T, origin='upper', extent=[0,1,1,0], cmap='inferno', aspect='auto')
    plt.colorbar(label='Jumlah titik lintasan'); plt.title('Heatmap kepadatan pergerakan')
    plt.xlabel('X ternormalisasi'); plt.ylabel('Y ternormalisasi'); plt.show()

## 3. Contoh lintasan per ID
Tampilkan beberapa track dengan minimal tiga titik. ID tracker centroid hanya berlaku di satu sesi kamera.

In [ ]:
if tracks.empty:
    print('Belum ada data lintasan.')
else:
    eligible = tracks.groupby(['session_id','track_id']).filter(lambda g: len(g) >= 3)
    selected = eligible.groupby(['session_id','track_id']).size().sort_values(ascending=False).head(12).index
    plt.figure(figsize=(9, 6))
    for session, track_id in selected:
        g = eligible[(eligible['session_id']==session) & (eligible['track_id']==track_id)].sort_values('sampled_at')
        plt.plot(g['x_norm'], g['y_norm'], marker='.', linewidth=1, label=f'{str(session)[:5]} / ID {track_id}')
    plt.gca().invert_yaxis(); plt.xlim(0,1); plt.ylim(1,0); plt.grid(alpha=.2)
    plt.title('Lintasan orang berdasarkan ID tracker centroid'); plt.xlabel('X ternormalisasi'); plt.ylabel('Y ternormalisasi')
    if len(selected) <= 12: plt.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
    plt.tight_layout(); plt.show()

## 4. Evaluasi jumlah orang dari label aktual
Kolom `actual_count` diisi melalui panel evaluasi dashboard. Baris yang belum dikonfirmasi tidak dihitung sebagai benar atau salah.

In [ ]:
if evaluation.empty or 'actual_count' not in evaluation:
    print('Belum ada sampel evaluasi.')
else:
    checked = evaluation.dropna(subset=['actual_count']).copy()
    if checked.empty:
        print('Belum ada jumlah aktual yang dikonfirmasi. Isi ulasan sampel di dashboard terlebih dahulu.')
    else:
        checked['actual_count'] = checked['actual_count'].astype(int)
        checked['error'] = checked['predicted_count'] - checked['actual_count']
        exact = (checked['error'] == 0).mean() * 100
        mae = checked['error'].abs().mean()
        rmse = np.sqrt((checked['error'] ** 2).mean())
        print(f'Sampel ditinjau: {len(checked)}')
        print(f'Ketepatan jumlah persis: {exact:.1f}% | MAE: {mae:.2f} orang | RMSE: {rmse:.2f} orang')
        display(checked[['captured_at','predicted_count','actual_count','error','confidence']].tail(20))

## 6. Frame gelap dan audio noise/quiet
Artikel memakai ambang brightness rata-rata <10 untuk menandai frame gelap. Aplikasi ini mencatat waktunya dan melewati deteksi agar frame gelap tidak dianggap sebagai ruangan kosong. Untuk audio, aplikasi hanya menyimpan level, zero-crossing, dan envelope yang telah direduksi; PCM mentah dibuang.

In [ ]:
import json
if quality_events.empty:
    print('Belum ada event kualitas frame gelap.')
else:
    dark = quality_events[quality_events['event_type']=='dark_frame']
    print(f'Jumlah episode frame gelap: {len(dark)}')
    if not dark.empty:
        plt.figure(figsize=(12,3)); plt.plot(dark['observed_at'],dark['brightness'],'.-'); plt.axhline(10,color='red',linestyle='--',label='ambang 10'); plt.ylabel('Brightness mean (0-255)'); plt.title('Episode frame gelap'); plt.legend(); plt.xticks(rotation=25); plt.tight_layout(); plt.show()

if audio.empty:
    print('Belum ada sampel audio. Periksa apakah sounddevice dan mikrofon tersedia.')
else:
    reviewed_audio = audio.dropna(subset=['actual_label']).copy()
    if reviewed_audio.empty:
        print('Belum ada label aktual noise/quiet. Tinjau sampel audio dari dashboard terlebih dahulu.')
    else:
        baseline = (reviewed_audio['predicted_label']==reviewed_audio['actual_label']).mean()*100
        print(f'Akurasi threshold yang tersimpan: {baseline:.1f}% pada {len(reviewed_audio)} label manual')
        from sklearn.metrics import classification_report, ConfusionMatrixDisplay
        print(classification_report(reviewed_audio['actual_label'], reviewed_audio['predicted_label'], labels=['quiet','noise'], zero_division=0))
        ConfusionMatrixDisplay.from_predictions(reviewed_audio['actual_label'], reviewed_audio['predicted_label'], labels=['quiet','noise'], cmap='Oranges'); plt.title('Confusion matrix - audio threshold'); plt.show()
        from sklearn.model_selection import train_test_split
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
        X=np.vstack(reviewed_audio['processed_features'].apply(json.loads).to_list())
        y=reviewed_audio['actual_label'].astype(str).to_numpy()
        class_counts=pd.Series(y).value_counts()
        if len(class_counts)==2 and class_counts.min()>=2 and len(y)>=10:
            X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,random_state=42,stratify=y)
            knn=KNeighborsClassifier(n_neighbors=min(5,len(y_train)),weights='distance')
            knn.fit(X_train,y_train); pred=knn.predict(X_test)
            print(f'k-NN holdout accuracy (70/30): {accuracy_score(y_test,pred):.3f}')
            print(classification_report(y_test,pred,zero_division=0))
            ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=['quiet','noise'],cmap='Oranges'); plt.title('Audio: quiet / noise'); plt.show()
        else:
            print('Butuh minimal 10 ulasan dan setidaknya 2 contoh untuk setiap kelas sebelum menguji k-NN 70/30.')


KNN holdout di notebook menggunakan label yang telah dikonfirmasi manual. Nilai ini baru layak dibandingkan setelah jumlah sampel tiap kelas cukup dan kondisi ruangan beragam. Hasil klasifikasi threshold pada dashboard adalah baseline, bukan akurasi model terlatih.

## 7. Durasi zona kerja dan pola masuk/keluar
Durasi kunjungan dihitung dari ID tracker saat titik tengah orang berada dalam ROI. ID dapat berubah saat terjadi oklusi, jadi hasil ini estimasi per kunjungan.

In [ ]:
if zone_visits.empty:
    print('Belum ada kunjungan zona kerja tersimpan.')
else:
    closed = zone_visits.dropna(subset=['duration_sec']).copy()
    print(f'Jumlah masuk tercatat: {len(zone_visits)} | kunjungan selesai: {len(closed)}')
    if not closed.empty:
        closed['duration_min'] = closed['duration_sec'] / 60
        display(closed['duration_min'].describe().to_frame('durasi_menit'))
        plt.figure(figsize=(10,3)); plt.hist(closed['duration_min'].clip(upper=60), bins=20, color='#f28c45'); plt.xlabel('Durasi per kunjungan (menit, dipotong 60)'); plt.ylabel('Jumlah kunjungan'); plt.title('Distribusi durasi zona'); plt.show()


## 5. Catatan interpretasi
- Heatmap menunjukkan lokasi kamera tempat titik pusat orang lebih sering tercatat; bukan peta koordinat ruangan yang terkalibrasi.
- ID dapat berubah saat orang tertutup, kamera terputus, atau sesi baru dimulai.
- Akurasi hanya representatif jika koreksi aktual dilakukan pada beragam waktu dan kondisi.
- Untuk melatih YOLO kembali, anotasi bounding box dan kelas objek pada gambar diperlukan; file jumlah orang saja tidak cukup.